In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/src/annotated/checkpoints/post_cell_10.pickle

In [ ]:
%%RecordEvent
%%time
### cell 11 ###

# cell 11 optimized
# Step 1: read all essays into Python lists (still CPU I/O)
ids = []
texts = []
for path in tqdm(train_txt):
    with open(path, 'r') as f:
        texts.append(f.read())
    # extract the "id" from the filename
    ids.append(path.rsplit('/', 1)[-1].replace('.txt', ''))

# Step 2: build a cudf DataFrame via the pandas API shim
# (the %load_ext cudf.pandas extension makes DataFrame() return a GPU-backed frame)
df_text = DataFrame({'id': ids, 'text': texts})

# Step 3: use GPU-accelerated string methods to compute lengths & word counts
df_text['essay_len']   = df_text['text'].str.strip().str.len()
df_text['essay_words'] = df_text['text'].str.split().list.len()

# Step 4: merge the new columns into `train` without reassigning `train`
tmp = train.merge(
    df_text[['id', 'essay_len', 'essay_words']],
    on='id',
    how='left'
)
train['essay_len']   = tmp['essay_len']
train['essay_words'] = tmp['essay_words']

In [ ]:
%Checkpoint /home/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/src/rewritten/o4_mini_high/checkpoints/post_cell_11_try_4.pickle

In [ ]:
%PrintCellInfo opt_cell_exec_info

In [ ]:

with open("/home/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/src/opt_cell_exec_info_11_try_4.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[11], f)


In [ ]:
opt_output = Out.get(4)